In [ ]:
import nltk
import pandas as pd
import torch
from nltk.tokenize import sent_tokenize, word_tokenize
from sentence_transformers import SentenceTransformer, util


nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def elabora_corpus_bilingue(testo_en, testo_it, min_parole=6, max_parole=15, target_coppie=500, soglia_confidenza=0.75):

    def filtra_frasi(testo, lingua):
        frasi = sent_tokenize(testo, language=lingua)
        frasi_valide = []
        for frase in frasi:
            parole = [p for p in word_tokenize(frase, language=lingua) if p.isalnum()]

            if min_parole <= len(parole) <= max_parole:
                frasi_valide.append(frase)
        return frasi_valide


    frasi_en = filtra_frasi(testo_en, 'english')
    frasi_it = filtra_frasi(testo_it, 'italian')


    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    modello = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)


    emb_en = modello.encode(frasi_en, convert_to_tensor=True)
    emb_it = modello.encode(frasi_it, convert_to_tensor=True)


    matrice_similarita = util.cos_sim(emb_en, emb_it)

    coppie_estratte = []
    indici_it_assegnati = set()

    for i in range(len(frasi_en)):
        if len(coppie_estratte) >= target_coppie:
            break

        miglior_idx_it = matrice_similarita[i].argmax().item()
        punteggio = matrice_similarita[i][miglior_idx_it].item()

        if punteggio >= soglia_confidenza and miglior_idx_it not in indici_it_assegnati:
            coppie_estratte.append({
                "Inglese": frasi_en[i],
                "Italiano": frasi_it[miglior_idx_it],
                "Affidabilità": round(punteggio, 3)
            })
            indici_it_assegnati.add(miglior_idx_it)

    return pd.DataFrame(coppie_estratte)


with open('en-it.tmx.txt.en', 'r', encoding='utf-8') as file_en:
    text_en = file_en.read()

with open('en-it.tmx.txt.it', 'r', encoding='utf-8') as file_it:
    text_it = file_it.read()


df_allineato = elabora_corpus_bilingue(text_en, text_it, min_parole=6, max_parole=15, target_coppie=500)


display(df_allineato.head())


df_allineato.to_csv('frasi_allineate_6_15.csv', index=False, sep=';', encoding='utf-8-sig')



0. Caricamento dei file di testo in corso...
File caricati correttamente in memoria.
1. Estrazione e filtraggio delle frasi in corso...
Frasi valide trovate: 13540 (EN), 12612 (IT)
2. Caricamento del modello vettoriale su: CPU


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3. Calcolo delle rappresentazioni semantiche e allineamento...

Elaborazione completata. Totale coppie estratte: 500


,Inglese,Italiano,Affidabilità
0,The color and opacity of the highlight border.,I rettangolini nero e bianco reimpostano i col...,0.799
1,Accerciser is an interactive Python accessibil...,Accerciser è un esploratore interattivo di acc...,0.944
2,"In essence, Accerciser is a next generation at...","Essenzialmente, Accerciser è uno strumento at-...",0.859
3,Accerciser Main Window\nShows main window.,Finestra principale di Accerciser\nVisualizza ...,0.923
4,"The menu bar contains File, Edit, Bookmarks, V...","Un elenco corredato di informazioni sull'uso, ...",0.762



Il file 'frasi_allineate_6_15.csv' è stato salvato con successo e formattato per la lettura in Excel.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import csv


df_completo = pd.read_csv(
    'frasi_allineate.csv',
    sep=';',
    engine='python',
    on_bad_lines='skip',
    quoting=csv.QUOTE_MINIMAL
)


df_train, df_test = train_test_split(df_completo, test_size=0.20, random_state=42)


df_train.to_csv('train_bilingue.csv', index=False, sep=';', encoding='utf-8-sig')
df_test.to_csv('test_bilingue.csv', index=False, sep=';', encoding='utf-8-sig')


df_train['Inglese'].to_csv('train_en.txt', index=False, header=False, encoding='utf-8')
df_test['Inglese'].to_csv('test_en.txt', index=False, header=False, encoding='utf-8')


df_train['Italiano'].to_csv('train_it.txt', index=False, header=False, encoding='utf-8')
df_test['Italiano'].to_csv('test_it.txt', index=False, header=False, encoding='utf-8')

--- INIZIO ELABORAZIONE E PARTIZIONE DEL DATASET ---
Dataset caricato. Totale coppie valide: 500
Dimensione Training Set: 400 coppie
Dimensione Test Set: 100 coppie
- File bilingui (.csv) salvati e formattati per Excel.
- File monolingua in Inglese (.txt) salvati.
- File monolingua in Italiano (.txt) salvati.

--- OPERAZIONE COMPLETATA CON SUCCESSO ---
